<a href="https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Provisional lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which content pages should an editor inspect first?  
**Goal of this notebook:** define and *verify* a small, safe data contract; build exactly five pre-decision features; demonstrate and remove one deliberately leaked field.

**Core first, AI second:** Review and personalize the writing. The code queries FlyRank's **gated** pseudonymized warehouse with DuckDB on Colab. A real run is required; this notebook does **not** contain invented warehouse results.

**Sources:** `skills/README.md` → `writing-data-contracts/SKILL.md`, plus `flyrank/flyrank-data/SKILL.md`. Do not put tokens, raw identifiers, URLs, or dataset exports in your public repository.

## 1. Unit of analysis + time window — my five plain-language contract answers

1. **One row means:** For my final *feature table*, one row is **one pseudonymized content item belonging to one pseudonymized client** at the **March 15, 2026 decision point**. The source `fact_content_daily_performance` is more granular: one `report_date` × `client_hash_id` × `content_hash_id` per daily record. I aggregate the daily source before modeling.
2. **Which table:** `fact_content_daily_performance` from FlyRank's gated Hugging Face warehouse, restricted to its **`month=2026-03` Parquet partition**. No joins or raw personal content are needed for this small exercise.
3. **Which window:** **March 1–15, 2026** provides five input features known when March 15 ends. **March 16–31** is an *outcome-only window* to assess later decline. I require at least 10 GSC-available observed days in **each** part, and at least 100 pre-window impressions. This is a two-window teaching design—not a full 30-day prediction experiment or a final capstone contract.
4. **What I would rank / predict:** Rank pages for human refresh review using a score based on their risk of **lower impressions per observed day in March 16–31** versus March 1–15. The binary outcome is `1` when the latter rate is **more than 20% lower**, else `0`. The label is *observed later-period behavior*, **not a measure of whether editing would have helped**. My evaluation metric is **Precision@50** on clients not used for model fitting.
5. **One thing deliberately excluded:** Any **post-March-15 value** (including the late-March impression total and the resulting decline label) is **never a legitimate model input**. Client and content IDs are context/grouping keys only. The leak experiment below violates this on purpose for illustration, then removes the leaky field.

**Human output:** a ranked **review recommendation** for an editor/SEO analyst; never automatic editing or publishing. A wrong priority wastes editorial effort or overlooks a page worth reviewing.

### Secure Colab setup — read before running

1. Sign in to [the FlyRank warehouse dataset](https://huggingface.co/datasets/FlyRank/internship-warehouse) and accept the data-use gate **in the browser first**.
2. Create a **Read** token at [Hugging Face → Access Tokens](https://huggingface.co/settings/tokens). A fine-grained token must explicitly allow gated-repository access; the plain Read type is simpler.
3. In Colab open the left **🔑 Secrets** panel, add `HF_TOKEN`, and enable notebook access. **Never paste the token in a code cell or commit it.**
4. Then run cells top to bottom. No BigQuery account is needed. Only March's partition is queried; the June `_sample` remains untouched.

If a query returns `401` or `403`, check that your Hugging Face account accepted the dataset gate and that the same account issued the token. For `429`, stop repeated queries and retry later.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [2]:
# Secure authentication and March partition setup (not a verification query).
import os
import re
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    raise RuntimeError("Add a Hugging Face READ token to Colab Secrets as HF_TOKEN and enable notebook access.")
if not re.fullmatch(r"hf_[A-Za-z0-9]+", hf_token.strip()):
    raise RuntimeError("HF_TOKEN doesn't look like a valid Hugging Face token. Check the Colab Secret.")

con = duckdb.connect()
try:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token.strip()}')")
except Exception:
    raise RuntimeError("Could not initialize Hugging Face access; check your DuckDB and token setup.") from None
finally:
    # No token is printed or stored in the notebook source/outputs.
    del hf_token

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print("DuckDB initialized for March 2026. Run the three contract checks below.")

DuckDB initialized for March 2026. Run the three contract checks below.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Five candidate features** | `impressions_pre_15d`, `clicks_pre_15d`, `active_days_pre_15d`, `avg_position_pre_15d`, `ctr_pre_pct_15d` | Aggregated from March 1–15 and therefore observable **before** the March 16 outcome window (assuming data was available at the decision point) |
| **Outcome label and inputs** | `decline_label`, `impressions_post_16d`, `pre_days`, `post_days` | For defining/testing the later-period outcome and auditing observation coverage; **not model features** |
| **Context / grouping** | `client_hash_id`, `content_hash_id`, `report_date` | Keep page grain straight and hold clients out during evaluation; IDs must not enter the model |
| **Excluded** | June `_sample`; any late-March data in features; unaligned 90-day query-table aggregates; client identity; `ga4_sessions` without reliable availability | Avoid future leakage, overlapping windows, identity exposure, and missing-as-zero mistakes |

**Missingness:** `gsc_data_available IS TRUE` is required before treating search measurements as observed; `ga4_data_available IS TRUE` is separate and **not** required for this GSC-only five-feature exercise. In particular, `IS NOT TRUE` includes both `FALSE` and `NULL`. I will *inspect* availability before filtering. `gsc_avg_position` can be null or zero (no valid position); median imputation for this exploratory exercise is fitted **only on the training clients**.

## 3. Verify with real SQL — exactly three verification queries

These queries run only on the **March 2026** Parquet partition. Keep their live outputs visible in the committed notebook. The following feature-construction SQL is **not** counted among these three verification queries.

**Verification 1 — daily grain:** `(report_date, client_hash_id, content_hash_id)` must be unique. Expect **zero duplicated key groups**. This is an explicit check, not an assumption.

In [3]:
# Verification query 1/3 — daily grain (no identifiers displayed).
q1 = f"""
SELECT COUNT(*) AS duplicated_daily_keys
FROM (
    SELECT report_date, client_hash_id, content_hash_id
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
) duplicates
"""
grain_check = con.sql(q1).df()
display(grain_check)
assert int(grain_check.loc[0, "duplicated_daily_keys"]) == 0, "Grain failed: investigate duplicates before proceeding."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicated_daily_keys
0,0


**Verification 2 — count and date span:** Measure **actual** March rows, earliest/latest observed date, and number of pseudonymized clients. The results are aggregated, not client-identifying.

In [4]:
# Verification query 2/3 — row count and date span.
q2 = f"""
SELECT COUNT(*) AS march_rows,
       MIN(report_date) AS earliest_date,
       MAX(report_date) AS latest_date,
       COUNT(DISTINCT client_hash_id) AS distinct_clients
FROM {MARCH}
"""
window_check = con.sql(q2).df()
display(window_check)
assert int(window_check.loc[0, "march_rows"]) > 0, "No March rows returned."

,march_rows,earliest_date,latest_date,distinct_clients
0,9841378,2026-03-01,2026-03-31,55


**Verification 3 — availability with `IS TRUE`:** Count how many records genuinely have GSC measurements, GA4 measurements, and both. Our proposed lane uses GSC only; the GA4 count shows why treating missing tracking as zero would be unsafe.

In [5]:
# Verification query 3/3 — three-valued availability flags.
q3 = f"""
SELECT COUNT(*) AS all_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                          AND ga4_data_available IS TRUE) AS both_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                          AND gsc_impressions IS NOT NULL) AS gsc_observed_impression_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                          AND gsc_avg_position IS NULL) AS gsc_missing_position_rows
FROM {MARCH}
"""
availability_check = con.sql(q3).df()
display(availability_check)
print("GSC-available rows retained:", f"{int(availability_check.loc[0, 'gsc_true_rows']):,}")
assert int(availability_check.loc[0, "gsc_true_rows"]) > 0, "No GSC-available rows."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,gsc_true_rows,ga4_true_rows,both_true_rows,gsc_observed_impression_rows,gsc_missing_position_rows
0,9841378,3611061,413966,364347,3611061,0


GSC-available rows retained: 3,611,061


### Build exactly five pre-decision features (separate from verification queries)

**Decision point:** end of March 15, 2026. Each feature is available then because it uses **only observations dated March 1–15**:

1. `impressions_pre_15d` — sum of GSC impressions on or before March 15; captures pre-window visibility.
2. `clicks_pre_15d` — sum of GSC clicks in that same past window; captures pre-window click activity.
3. `active_days_pre_15d` — number of GSC-available early-March days with at least one impression; captures consistency of visibility.
4. `avg_position_pre_15d` — mean of **valid positive** average positions in the pre-window; zero and null do not represent a genuine top rank.
5. `ctr_pre_pct_15d` — 100 × past clicks / past impressions, expressed in **percent units** (e.g. 0.76 means 0.76%).

**Later outcome, not a feature:** did the observed GSC impressions **per GSC-available day** fall by more than 20% during March 16–31, compared with March 1–15? Each half requires ≥10 available days; ≥100 early impressions stabilizes the comparison. The selection and threshold are **provisional** and may bias this first exercise toward well-observed pages. Treat this as a small demonstration, not a production-ready or temporally validated model.

In [6]:
# Feature-building query (distinct from the EXACTLY THREE verification queries).
# Aggregate March daily facts per client/content item before converting to pandas.
feature_sql = f"""
WITH per_page AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (
               WHERE report_date < DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS pre_days,
           COUNT(*) FILTER (
               WHERE report_date >= DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS post_days,
           SUM(gsc_impressions) FILTER (
               WHERE report_date < DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS impressions_pre_15d,
           SUM(gsc_clicks) FILTER (
               WHERE report_date < DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS clicks_pre_15d,
           COUNT(*) FILTER (
               WHERE report_date < DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
                 AND gsc_impressions > 0
           ) AS active_days_pre_15d,
           AVG(NULLIF(gsc_avg_position, 0)) FILTER (
               WHERE report_date < DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS avg_position_pre_15d,
           SUM(gsc_impressions) FILTER (
               WHERE report_date >= DATE '2026-03-16'
                 AND gsc_data_available IS TRUE
           ) AS impressions_post_16d
    FROM {MARCH}
    GROUP BY client_hash_id, content_hash_id
)
SELECT client_hash_id, content_hash_id,
       pre_days, post_days,
       impressions_pre_15d, clicks_pre_15d, active_days_pre_15d,
       avg_position_pre_15d,
       100.0 * clicks_pre_15d / NULLIF(impressions_pre_15d, 0) AS ctr_pre_pct_15d,
       impressions_post_16d,
       CAST(
           (impressions_post_16d * 1.0 / NULLIF(post_days, 0))
           < 0.8 * (impressions_pre_15d * 1.0 / NULLIF(pre_days, 0))
           AS INTEGER
       ) AS decline_label
FROM per_page
WHERE pre_days >= 10 AND post_days >= 10
  AND impressions_pre_15d >= 100
  AND impressions_post_16d IS NOT NULL
"""
frame = con.sql(feature_sql).df()
FEATURES = [
    "impressions_pre_15d", "clicks_pre_15d", "active_days_pre_15d",
    "avg_position_pre_15d", "ctr_pre_pct_15d",
]
assert len(FEATURES) == 5
assert frame[["client_hash_id", "content_hash_id"]].duplicated().sum() == 0
assert frame["decline_label"].notna().all(), "Unexpected missing outcomes."
assert len(frame) >= 100, "Too few eligible items; inspect availability and window thresholds."
print("Eligible page-level rows:", f"{len(frame):,}")
print("Observed later-period decline rate:", f"{frame['decline_label'].mean():.1%}")
print("Missing values in the FIVE candidate features:")
display(frame[FEATURES].isna().sum().rename("missing_rows").to_frame())
# Do NOT display content/client IDs or per-page outcomes.
display(frame[FEATURES].head(8).reset_index(drop=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible page-level rows: 73,539
Observed later-period decline rate: 34.3%
Missing values in the FIVE candidate features:


,missing_rows
impressions_pre_15d,0
clicks_pre_15d,0
active_days_pre_15d,0
avg_position_pre_15d,0
ctr_pre_pct_15d,0


,impressions_pre_15d,clicks_pre_15d,active_days_pre_15d,avg_position_pre_15d,ctr_pre_pct_15d
0,429.0,2.0,15,4.247255,0.466200
1,628.0,1.0,15,5.330069,0.159236
2,1280.0,9.0,15,4.468441,0.703125
3,2531.0,10.0,15,4.907404,0.395101
4,243.0,1.0,15,14.534282,0.411523
5,176.0,0.0,15,36.527344,0.000000
6,2217.0,7.0,15,8.925289,0.315742
7,129.0,0.0,15,7.432614,0.000000


### The leakage trap — compare, then REMOVE it

This is a **controlled anti-example**, not an acceptable feature. We deliberately copy the outcome (`decline_label`) into **one** fake model input called `leaked_target_copy`. This makes the answer directly available to a model and should produce an implausibly strong result. Then we remove the leaky field and retain the honest five-feature score.

For a modest demonstration, hold out **entire clients** with `GroupShuffleSplit` and calculate **Precision@50** and **ROC-AUC**; learn the imputation median on training clients only. Even client-holdout cannot establish month-to-month generalization; that requires a later time-split evaluation.

In [7]:
# One leak only; same client-holdout partition and model for fair comparison.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

y = frame["decline_label"].astype(int)
groups = frame["client_hash_id"]  # split-only context; never a model feature
split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(split.split(frame[FEATURES], y, groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert len(test_idx) >= 50, "Need at least 50 test pages for Precision@50."
assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2, "Need both classes in both partitions."

def quick_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=80, max_depth=6, min_samples_leaf=10,
            random_state=42, n_jobs=-1
        ),
    )

def precision_50(y_true, scores):
    indices = np.argsort(-np.asarray(scores), kind="stable")[:50]
    return float(np.asarray(y_true)[indices].mean())

X_honest = frame[FEATURES].copy()
honest_model = quick_model()
honest_model.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
honest_scores = honest_model.predict_proba(X_honest.iloc[test_idx])[:, 1]
honest_p50 = precision_50(y.iloc[test_idx], honest_scores)
honest_auc = roc_auc_score(y.iloc[test_idx], honest_scores)

# INTENTIONAL LEAK: directly duplicates the observed label; never deploy this.
frame["leaked_target_copy"] = frame["decline_label"]
X_leaky = frame[FEATURES + ["leaked_target_copy"]].copy()
leaky_model = quick_model()
leaky_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_scores = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_p50 = precision_50(y.iloc[test_idx], leaky_scores)
leaky_auc = roc_auc_score(y.iloc[test_idx], leaky_scores)

print("Client-holdout demonstration (NOT time-forward validation)")
print(f"Honest five-feature score: P@50 = {honest_p50:.3f} | ROC-AUC = {honest_auc:.3f}")
print(f"With leaked label-copy:    P@50 = {leaky_p50:.3f} | ROC-AUC = {leaky_auc:.3f}")
print("The leaked result is INVALID because it copies the answer into the inputs.")

# REMOVE the single label-derived feature from all working frames.
frame.drop(columns=["leaked_target_copy"], inplace=True)
X_leaky.drop(columns=["leaked_target_copy"], inplace=True)
del leaky_model, leaky_scores
assert "leaked_target_copy" not in frame.columns
assert list(X_honest.columns) == FEATURES == list(X_leaky.columns)
print(f"Leakage removed. Retained HONEST five-feature P@50 = {honest_p50:.3f}")

Client-holdout demonstration (NOT time-forward validation)
Honest five-feature score: P@50 = 0.560 | ROC-AUC = 0.565
With leaked label-copy:    P@50 = 1.000 | ROC-AUC = 1.000
The leaked result is INVALID because it copies the answer into the inputs.
Leakage removed. Retained HONEST five-feature P@50 = 0.560


## 4. One important data limitation

The warehouse is an **unbalanced panel**: clients and pages have different durations of observed history. A missing GSC/GA4 measurement is not automatically zero activity; reporting and access start dates differ. Restricting this exercise to `gsc_data_available IS TRUE` and requiring both March halves reduces but **does not eliminate** measurement/selection bias. Our label measures later **impression decline**, not content-staleness or the causal benefit of an edit.

**What this can claim:** March's observed data supports a *directional, decision-support* ranking experiment. The deliberately leaky model is invalid; the five-feature result is a **single-client-holdout demonstration**, not independently verified future performance.

**Next steps (later assignments):** align feature/label windows across more months, check per-client history coverage, compare against a transparent ranking baseline on held-out clients/time, and let editors review recommendations before any content change.

## 5. Self-check — tick only after your real Colab run

- [ ] I can explain my **five contract answers**: row grain, warehouse table, March feature/outcome windows, decline label, and deliberate exclusion.
- [ ] I can point to **exactly three executed verification-query outputs**: uniqueness, row count/date span, and `IS TRUE` availability.
- [ ] I built **exactly five pre-decision features** and explained when each would have been knowable.
- [ ] I ran the **one-column leakage experiment**, observed the scores, and confirmed that I **removed** the leaked column.
- [ ] I can explain the unbalanced-panel / access-availability limitation and why these findings are not causal proof.
- [ ] I reviewed the wording in my own words, checked all notebook cells execute without errors, and did not put tokens/client identifiers/new datasets in GitHub.
- [ ] I saved the **executed notebook** to `work/notebooks/w03_data_contract.ipynb` on branch `main` and verified all outputs on github.com.

**Final deliverable:** submit only your public repository URL: `https://github.com/Jhonas05/flyrank-ml-internship`. Do **not** submit Colab or Drive links.